## **Semantic Distance Metric:**

In [ ]:
%pip install sentence_transformers -q --user

In [ ]:
# Free Alternative to using OpenAI Embeddings API
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
sentence = ['This blanket has such a cozy temperature for me!','I am so much warmer and snug using this spread!',
            'Tylor Swift eas 34 years old in 2024']

embedding = model.encode(sentence)
print(embedding)
embedding.shape

## **Distance Mertics:**

### Euclidean Distance(L2):

In [ ]:
def euclidean_distance(vec1, vec2):
    return np.linalg.norm(vec1 - vec2)

In [ ]:
# Low Score = High Similarity
print("Euclidean Distance: Review 1 vs Review 2:",
      euclidean_distance(embedding[0], embedding[1]))

print("Euclidean Distance: Review 1 vs Review random comment:",
      euclidean_distance(embedding[0], embedding[2]))

print("Euclidean Distance: Review 2 vs Review random comment:",
      euclidean_distance(embedding[1], embedding[2]))

### Dot Product (Inner Product):

In [ ]:
# Positive and High Score = High Similarity
# Negative and Low Score = Low Similarity
print("Euclidean Distance: Review 1 vs Review 2:",
      np.dot(embedding[0], embedding[1]))

print("Euclidean Distance: Review 1 vs Review random comment:",
      np.dot(embedding[0], embedding[2]))

print("Euclidean Distance: Review 2 vs Review random comment:",
      np.dot(embedding[1], embedding[2]))

### Cosin Distance:

In [ ]:
def cosine_distance(vec1, vec2):
    cosine = 1 - abs((np.dot(vec1, vec2) / (np.linalg.norm(vec1)*np.linalg.norm(vec2))))
    return cosine

In [ ]:
# Low Score = High Similarity
print("Cosine Distance: Review 1 vs Review 2:",
      cosine_distance(embedding[0], embedding[1]))

print("Cosine Distance: Review 1 vs Review random comment:",
      cosine_distance(embedding[0], embedding[2]))

print("Cosine Distance: Review 2 vs Review random comment:",
      cosine_distance(embedding[1], embedding[2]))

## **Hybrid Custom:**

In [ ]:
%pip install --upgrade pip

# Uninstall conflicting packages
%pip uninstall -y langchain-core langchain-openai langchain-experimental langchain-community langchain chromadb beautifulsoup4 python-dotenv PyPDF2 rank_bm25

# Install compatible versions of langchain-core and langchain-openai
%pip install langchain-community==0.4.1
%pip install langchain-text-splitters==1.0.0
%pip install langchain-openai==1.1.0
%pip install langsmith==0.4.49
%pip install langchain==1.1.0

# Install remaining packages
%pip install langchain-chroma==1.0.0
%pip install chromadb==1.3.5
%pip install python-dotenv==1.2.1

# new
%pip install PyPDF2==3.0.1 -q --user
%pip install rank_bm25==0.2.2

# Restart the kernel after installation

In [ ]:
import os
os.environ['USER_AGENT'] = 'RAGUserAgent'
import openai
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langsmith import Client
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate

# new
from PyPDF2 import PdfReader
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

In [ ]:
# variables
_ = load_dotenv(dotenv_path='env.txt')
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
openai.api_key = os.environ['OPENAI_API_KEY']
embedding_function = OpenAIEmbeddings()
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
pdf_path = "google-2023-environmental-report.pdf"
collection_name = "google_environmental_report"
str_output_parser = StrOutputParser()
user_query = "What are Google's environmental initiatives?"

In [ ]:
#### INDEXING ####

In [ ]:
# Load the PDF and extract text
pdf_reader = PdfReader(pdf_path)
text = ""
for page in pdf_reader.pages:
    text += page.extract_text()

In [ ]:
# Split
character_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=1000,
    chunk_overlap=200
)
splits = character_splitter.split_text(text)